In [ ]:
import os
from pyspark.sql import functions as F
from src.utils.config import *
from src.utils.kafka import create_kafka_consumer, insert_to_clickhouse
from src.utils.spark import get_spark

spark = get_spark()

save_to_clickhouse = os.getenv('SAVE_TO_CLICKHOUSE')
boostrap_servers = os.getenv('KAFKA_BOOTSTRAP_SERVERS')

ch_user = os.getenv('CLICKHOUSE_USER')
ch_password = os.getenv('CLICKHOUSE_PW')
ch_host = os.getenv('CLICKHOUSE_HOST')
client = setup_clickhouse_client(ch_user, ch_password, ch_host)

consumer_instance = create_kafka_consumer(
    client_id='flight_processor',
    bootstrap_servers=boostrap_servers,
    topic_name='raw_aviation_data',
)

In [ ]:
col_names=['flight_type', 'status', 'iata_number', 'airline_name', 'dep_scheduled_time', 'dep_actual_time', 'dep_delay_mins', 
           'arr_scheduled_time', 'arr_actual_time', 'arr_delay_mins',  'codeshared_airline', 'codeshared_flight_number']

defaults = {
    "flight_type": "Nil",
    "status": "Unknown",
    "iata_number": "",
    "airline_name": "",
    "dep_scheduled_time": "2099-12-31t00:00:00.000",
    "dep_actual_time": "2099-12-31t00:00:00.000",
    "dep_delay_mins": 0,
    "arr_scheduled_time": "2099-12-31t00:00:00.000",
    "arr_actual_time": "2099-12-31t00:00:00.000",
    "arr_delay_mins": 0,
    "codeshared_airline": "",
    "codeshared_flight_number": ""
}

insert_to_clickhouse(consumer_instance, 'historical_flight_data', client, save_to_clickhouse=False)